# Extração Scopus (pybliometrics) — Produção Bibliográfica

Notebook original (`Quick-Start.ipynb`) era a demonstração padrão da biblioteca
[`pyscopus`](http://zhiyzuo.github.io/python-scopus/). Essa lib está
desatualizada e vinha causando problemas, então esta versão migra para
[`pybliometrics`](https://github.com/pybliometrics-dev/pybliometrics) — wrapper
oficial e ativamente mantido para as APIs Scopus/ScienceDirect/SciVal da Elsevier.

O pipeline continua o mesmo: dada uma lista de pessoas (com `id_lattes` e
`scopus_author_id`), para cada uma buscamos todas as publicações indexadas na
Scopus e produzimos DataFrames **no mesmo formato de saída do `analyse.ipynb`**:

- `df_artigos_periodico_scopus` → mesmo schema de `df_artigos_final` (artigos de periódico)
- `df_artigos_congresso_scopus` → mesmo schema de `df_artigos_congresso_final` (trabalhos de congresso/proceedings)

Diferente da versão com `pyscopus`, a classe `ScopusSearch` do `pybliometrics`
já retorna, em uma única chamada por pessoa, `doi`, `author_names`, `issn`,
`pageRange` e `aggregationType` — não é mais necessário um segundo request por
publicação só para complementar DOI/ISSN.

## 1. Instalação e Configuração

In [2]:
%pip install pybliometrics python-dotenv

  Using cached pybliometrics-4.4.1-py3-none-any.whl.metadata (5.5 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
Using cached pybliometrics-4.4.1-py3-none-any.whl (126 kB)
Using cached tqdm-4.68.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pybliometrics]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import pybliometrics
from pybliometrics.scopus import ScopusSearch
#from pybliometrics.scopus.exception import Scopus401Error, ScopusQueryError

load_dotenv()

# Sua API Key da Elsevier (https://dev.elsevier.com/) — registre uma chave e
# guarde em uma variável de ambiente SCOPUS_API_KEY (ex.: em um arquivo .env).
# Na primeira execução, se nenhuma chave for encontrada, o pybliometrics.init()
# pede a chave interativamente e cria o arquivo de configuração em
# ~/.config/pybliometrics.cfg (não é necessário repetir isso depois).
API_KEY = os.getenv('SCOPUS_API_KEY')
INST_TOKEN = os.getenv('SCOPUS_INST_TOKEN')  # opcional, só se sua instituição usar token

if API_KEY:
    pybliometrics.init(keys=[API_KEY], inst_tokens=[INST_TOKEN] if INST_TOKEN else None)
else:
    # Lê o arquivo de configuração já existente (~/.config/pybliometrics.cfg)
    # ou pede a chave de forma interativa caso ainda não exista
    pybliometrics.init()

print("Cliente Scopus (pybliometrics) inicializado com sucesso!")

Creating config file at /home/alobogiron/.config/pybliometrics.cfg with default paths...
Configuration file successfully created at /home/alobogiron/.config/pybliometrics.cfg
For details see https://pybliometrics.rtfd.io/en/stable/configuration.html.
Cliente Scopus (pybliometrics) inicializado com sucesso!


## 2. Lista de pessoas a extrair

Cada pessoa precisa ter:

- `id_lattes`: a chave que une todas as fontes (Lattes, ORCID, Scopus) — é o que vai virar a FK no banco.
- `scopus_author_id`: o Author ID da Scopus da pessoa (ex.: `'57189222659'`), usado só para consultar a API.

Substitua a lista de exemplo abaixo pela sua lista real.

In [2]:
# Exemplo de lista de pessoas. Troque por:
#   df_pessoas_lista = pd.read_csv('lista_pessoas.csv')  # colunas: id_lattes, orcid_id, scopus_author_id
lista_pessoas_scopus = [
    {'id_lattes': '0000000000000001', 'scopus_author_id': '6603459594'},
    # {'id_lattes': '...', 'scopus_author_id': '...'},
]

print(f"Total de pessoas a processar: {len(lista_pessoas_scopus)}")

Total de pessoas a processar: 1


## 3. Extração das publicações de cada autor

Para cada pessoa usamos `ScopusSearch('AU-ID(<scopus_author_id>)')`, que devolve
em `.results` uma lista de namedtuples — uma por publicação — já contendo
`title`, `publicationName`, `coverDate`, `doi`, `issn`, `pageRange`,
`author_names`, `aggregationType`, entre outros.

Classificamos cada publicação em **periódico** ou **congresso/proceedings**
usando `aggregationType`:

- `'Journal'` (e afins: `'Book Series'`) → artigo de periódico
- `'Conference Proceeding'` → trabalho de congresso

> **Nota sobre limites/cache da API:** o `pybliometrics` já armazena em cache
> local os resultados de cada `ScopusSearch`, então execuções repetidas não
> reconsultam a API (use `refresh=True` se quiser forçar atualização). Ainda
> assim, mantemos um pequeno `time.sleep` entre pessoas para respeitar o
> rate-limit da sua chave.

In [4]:
def extrair_ano(cover_date):
    """Extrai o ano (int) de uma string de data tipo '2018-08-01'."""
    if not cover_date or pd.isna(cover_date):
        return pd.NA
    try:
        return int(str(cover_date)[:4])
    except (ValueError, TypeError):
        return pd.NA


lista_artigos_periodico_scopus = []
lista_artigos_congresso_scopus = []

TIPOS_PERIODICO_SCOPUS = {'journal', 'book series'}
TIPOS_CONGRESSO_SCOPUS = {'conference proceeding'}

for pessoa in lista_pessoas_scopus:
    id_lattes = pessoa['id_lattes']
    scopus_author_id = pessoa['scopus_author_id']

    print(f"Processando Scopus Author ID {scopus_author_id} (id_lattes={id_lattes})...")

    try:
        s = ScopusSearch(f'AU-ID({scopus_author_id})')
    except () as exc:
        print(f"  -> Falha ao buscar publicações de {scopus_author_id}: {exc}")
        continue
    except Exception as exc:
        print(f"  -> Erro inesperado em {scopus_author_id}: {exc}")
        continue

    publicacoes = s.results or []
    if not publicacoes:
        print("  -> Nenhuma publicação encontrada.")
        continue

    for pub in publicacoes:
        titulo = pub.title or pd.NA
        revista = pub.publicationName or pd.NA
        ano = extrair_ano(pub.coverDate)
        doi = pub.doi or pd.NA
        issn = pub.issn or pd.NA
        autores = pub.author_names or pd.NA
        tipo_agregacao = (pub.aggregationType or '').strip().lower()

        if tipo_agregacao in TIPOS_CONGRESSO_SCOPUS:
            # --- Trabalho de Congresso / Proceedings ---
            lista_artigos_congresso_scopus.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'ano': ano,
                'doi': doi,
                'autores': autores,
                'titulo_evento_lattes': revista,
                'paginas': pub.pageRange or pd.NA,
                'sigla_evento_google': pd.NA,
                'titulo_evento_google': pd.NA,
                'estrato': pd.NA,
                'tipo_match': 'SCOPUS',
                'coautoria_aluno': pd.NA,
            })
        else:
            # --- Artigo de Periódico (default para os demais aggregationType) ---
            lista_artigos_periodico_scopus.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'titulo_revista_lattes': pd.NA,  # não há contraparte do Lattes aqui
                'ano_pub': ano,
                'doi': doi,
                'autores': autores,
                'match_adequado': pd.NA,
                'coautoria_aluno': pd.NA,
                'id_scopus': pub.eid or pd.NA,
                'titulo_revista_scopus': revista,
                'maior_percentil': pd.NA,
                'codigo_area_maior_percentil': pd.NA,
                'area_maior_percentil': pd.NA,
                'issn': issn,
                'computation_area': pd.NA,
            })

    time.sleep(0.2)

print("\nExtração concluída.")
print(f"Artigos de periódico extraídos da Scopus: {len(lista_artigos_periodico_scopus)}")
print(f"Trabalhos de congresso extraídos da Scopus: {len(lista_artigos_congresso_scopus)}")

Processando Scopus Author ID 6603459594 (id_lattes=0000000000000001)...
  -> Erro inesperado em 6603459594: The requestor is not authorized to access the requested view or fields of the resource

Extração concluída.
Artigos de periódico extraídos da Scopus: 0
Trabalhos de congresso extraídos da Scopus: 0


## 4. Consolidação nos DataFrames finais (mesmo schema do `analyse.ipynb`)

As colunas abaixo replicam exatamente:

- `df_artigos_final` → `id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, doi, autores, match_adequado, coautoria_aluno, id_scopus, titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil, area_maior_percentil, issn, computation_area`
- `df_artigos_congresso_final` → `id_lattes, titulo_artigo, ano, doi, autores, titulo_evento_lattes, paginas, sigla_evento_google, titulo_evento_google, estrato, tipo_match, coautoria_aluno`

Os campos que a Scopus não fornece nesta extração (`titulo_revista_lattes`,
`maior_percentil`, `area_maior_percentil`, `computation_area`, `match_adequado`,
`estrato`, `sigla_evento_google`/`titulo_evento_google`) ficam nulos — o
cruzamento de percentil/área e de eventos já é feito pelo próprio
`analyse.ipynb`; se quiser aplicar a mesma lógica aqui, basta reaproveitá-la
passando estes DataFrames no lugar de `df_bib_artigos`/`df_bib_trab_congresso`.

In [ ]:
colunas_periodico = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area',
]

colunas_congresso = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
]

df_artigos_periodico_scopus = pd.DataFrame(lista_artigos_periodico_scopus, columns=colunas_periodico)
df_artigos_congresso_scopus = pd.DataFrame(lista_artigos_congresso_scopus, columns=colunas_congresso)

# Mesma tipagem usada no analyse.ipynb para permitir o concat sem surpresas
if not df_artigos_periodico_scopus.empty:
    df_artigos_periodico_scopus['ano_pub'] = pd.to_numeric(df_artigos_periodico_scopus['ano_pub'], errors='coerce').astype('Int64')
    df_artigos_periodico_scopus['id_lattes'] = df_artigos_periodico_scopus['id_lattes'].astype(str)
    df_artigos_periodico_scopus['titulo_revista_scopus'] = (
        df_artigos_periodico_scopus['titulo_revista_scopus'].astype(str).str.upper().str.strip()
    )

if not df_artigos_congresso_scopus.empty:
    df_artigos_congresso_scopus['ano'] = pd.to_numeric(df_artigos_congresso_scopus['ano'], errors='coerce').astype('Int64')
    df_artigos_congresso_scopus['id_lattes'] = df_artigos_congresso_scopus['id_lattes'].astype(str)

print("=== df_artigos_periodico_scopus ===")
display(df_artigos_periodico_scopus.head())
df_artigos_periodico_scopus.info()

print("\n=== df_artigos_congresso_scopus ===")
display(df_artigos_congresso_scopus.head())
df_artigos_congresso_scopus.info()

## 5. Próximo passo: unificar com `analyse.ipynb` e `orcid.ipynb`

```python
df_periodicos_unificado = pd.concat(
    [df_artigos_final, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)

df_congressos_unificado = pd.concat(
    [df_artigos_congresso_final, df_artigos_congresso_orcid, df_artigos_congresso_scopus],
    ignore_index=True,
)
```

Esses DataFrames já estão no formato esperado por `tb_artigo_periodico` e
`tb_artigo_conferencia` no DuckDB (mesmo `INSERT INTO ... (...)` que o
`analyse.ipynb` já define), prontos para subir ao banco. Antes de inserir,
garanta que toda pessoa referenciada em `id_lattes` já exista em
`tb_professores` (a FK exige isso).

Opcionalmente, é possível remover duplicatas entre as fontes (ex.: o mesmo
artigo aparecendo no Lattes e na Scopus) usando `doi` como chave — quando o
`doi` é nulo, cair de volta para `(titulo_artigo, id_lattes)`:

```python
df_periodicos_unificado['chave_dedup'] = df_periodicos_unificado['doi'].fillna(
    df_periodicos_unificado['titulo_artigo'].str.upper().str.strip() + '|' + df_periodicos_unificado['id_lattes']
)
df_periodicos_unificado = df_periodicos_unificado.drop_duplicates(subset=['chave_dedup']).drop(columns=['chave_dedup'])
```